# 🎨 AI Content Generator MVP
### Text → Image → Video → Audio (All Free!)

**Features:**
- Text-to-Image (Stable Diffusion XL Turbo)
- Text-to-Video (AnimateDiff / ModelScope)
- Text-to-Speech (Bark)
- Combined Video + Audio output

**Setup:** Runtime → Change runtime type → T4 GPU

In [ ]:
#@title 1️⃣ Install Dependencies (Run Once)
!pip install -q diffusers transformers accelerate safetensors
!pip install -q gradio spaces
!pip install -q imageio[ffmpeg] moviepy
!pip install -q bark scipy
!pip install -q torch torchvision torchaudio
!pip install -q xformers
print("✅ All dependencies installed!")

In [ ]:
#@title 2️⃣ Check GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
#@title 3️⃣ Text-to-Image Module
from diffusers import AutoPipelineForText2Image
import torch

# Using SDXL-Turbo for fast generation (4 steps!)
image_pipe = None

def load_image_model():
    global image_pipe
    if image_pipe is None:
        print("Loading SDXL-Turbo...")
        image_pipe = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/sdxl-turbo",
            torch_dtype=torch.float16,
            variant="fp16"
        )
        image_pipe.to("cuda")
        print("✅ Image model loaded!")
    return image_pipe

def generate_image(prompt, num_steps=4, guidance_scale=0.0):
    """Generate image from text prompt"""
    pipe = load_image_model()
    image = pipe(
        prompt=prompt,
        num_inference_steps=num_steps,
        guidance_scale=guidance_scale
    ).images[0]
    return image

# Test it
load_image_model()
test_image = generate_image("A beautiful sunset over mountains, cinematic lighting")
display(test_image)

In [ ]:
#@title 4️⃣ Text-to-Video Module
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video
import torch

video_pipe = None

def load_video_model():
    global video_pipe
    if video_pipe is None:
        print("Loading Text-to-Video model...")
        video_pipe = DiffusionPipeline.from_pretrained(
            "damo-vilab/text-to-video-ms-1.7b",
            torch_dtype=torch.float16,
            variant="fp16"
        )
        video_pipe.to("cuda")
        # Enable memory optimizations
        video_pipe.enable_model_cpu_offload()
        video_pipe.enable_vae_slicing()
        print("✅ Video model loaded!")
    return video_pipe

def generate_video(prompt, num_frames=16, num_steps=25):
    """Generate video from text prompt"""
    pipe = load_video_model()
    video_frames = pipe(
        prompt=prompt,
        num_frames=num_frames,
        num_inference_steps=num_steps
    ).frames[0]
    
    # Export to file
    output_path = "/content/output_video.mp4"
    export_to_video(video_frames, output_path, fps=8)
    return output_path, video_frames

# Test it (takes ~2-3 minutes)
print("Generating video... (this takes a few minutes)")
video_path, frames = generate_video("A cat walking in a garden")
print(f"✅ Video saved to: {video_path}")

# Display in notebook
from IPython.display import Video
Video(video_path, embed=True)

In [ ]:
#@title 5️⃣ Text-to-Speech Module (Bark)
from bark import SAMPLE_RATE, generate_audio, preload_models
from scipy.io.wavfile import write as write_wav
import numpy as np

def load_audio_model():
    print("Loading Bark TTS model...")
    preload_models()
    print("✅ Audio model loaded!")

def generate_speech(text, output_path="/content/output_audio.wav"):
    """Generate speech from text"""
    # Generate audio
    audio_array = generate_audio(text)
    
    # Save to file
    write_wav(output_path, SAMPLE_RATE, audio_array)
    return output_path, audio_array

# Test it
load_audio_model()
audio_path, _ = generate_speech("Hello! This is an AI generated voice. Pretty cool, right?")
print(f"✅ Audio saved to: {audio_path}")

# Play in notebook
from IPython.display import Audio
Audio(audio_path)

In [ ]:
#@title 6️⃣ Combine Video + Audio
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

def combine_video_audio(video_path, audio_path, output_path="/content/final_output.mp4"):
    """Combine video with audio track"""
    # Load video and audio
    video = VideoFileClip(video_path)
    audio = AudioFileClip(audio_path)
    
    # Adjust audio duration to match video
    if audio.duration > video.duration:
        audio = audio.subclip(0, video.duration)
    
    # Combine
    final_video = video.set_audio(audio)
    final_video.write_videofile(output_path, codec='libx264', audio_codec='aac')
    
    # Cleanup
    video.close()
    audio.close()
    
    return output_path

# Test combination
final_path = combine_video_audio("/content/output_video.mp4", "/content/output_audio.wav")
print(f"✅ Final video with audio: {final_path}")
Video(final_path, embed=True)

In [ ]:
#@title 7️⃣ 🚀 Launch Gradio UI (Full App)
import gradio as gr
import gc

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

def full_pipeline(prompt, voice_text, generate_video_flag=True):
    """Complete pipeline: Text → Image/Video → Audio → Combined"""
    outputs = {}
    
    # Generate Image
    yield "Generating image...", None, None, None
    image = generate_image(prompt)
    image.save("/content/output_image.png")
    outputs['image'] = "/content/output_image.png"
    
    if generate_video_flag:
        # Free up memory
        clear_memory()
        
        # Generate Video
        yield "Generating video (this takes a few minutes)...", image, None, None
        video_path, _ = generate_video(prompt, num_frames=16)
        outputs['video'] = video_path
        
        # Free up memory again
        clear_memory()
    
    # Generate Audio
    yield "Generating audio...", image, outputs.get('video'), None
    if voice_text.strip():
        audio_path, _ = generate_speech(voice_text)
        outputs['audio'] = audio_path
        
        # Combine if video exists
        if 'video' in outputs:
            yield "Combining video and audio...", image, outputs['video'], audio_path
            final_path = combine_video_audio(outputs['video'], audio_path)
            outputs['final'] = final_path
    
    final_video = outputs.get('final', outputs.get('video'))
    final_audio = outputs.get('audio')
    
    yield "✅ Done!", image, final_video, final_audio

# Create Gradio Interface
with gr.Blocks(title="AI Content Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎨 AI Content Generator MVP")
    gr.Markdown("Generate images, videos, and audio from text - all free!")
    
    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(
                label="Visual Prompt",
                placeholder="A futuristic city at night with neon lights...",
                lines=3
            )
            voice_input = gr.Textbox(
                label="Voice/Narration Text (optional)",
                placeholder="Welcome to the future...",
                lines=2
            )
            video_checkbox = gr.Checkbox(label="Generate Video (slower)", value=True)
            generate_btn = gr.Button("🚀 Generate", variant="primary")
            status = gr.Textbox(label="Status", interactive=False)
        
        with gr.Column():
            image_output = gr.Image(label="Generated Image")
            video_output = gr.Video(label="Generated Video")
            audio_output = gr.Audio(label="Generated Audio")
    
    generate_btn.click(
        full_pipeline,
        inputs=[prompt_input, voice_input, video_checkbox],
        outputs=[status, image_output, video_output, audio_output]
    )
    
    gr.Markdown("### 💡 Tips")
    gr.Markdown("""
    - **Image only**: Uncheck 'Generate Video' for faster results
    - **Video**: Takes 2-5 minutes on T4 GPU
    - **Audio**: Keep narration under 30 seconds
    """)

# Launch with public URL
demo.queue().launch(share=True, debug=True)

In [ ]:
#@title 📥 Download Your Creations
from google.colab import files

# Download all generated files
import os

for filename in ['output_image.png', 'output_video.mp4', 'output_audio.wav', 'final_output.mp4']:
    filepath = f'/content/{filename}'
    if os.path.exists(filepath):
        print(f"Downloading {filename}...")
        files.download(filepath)